# CrisisLens Libya — MARBERT Full Training Pipeline

## SIC Capstone Notebook

This notebook documents the **complete training lineage** of the CrisisLens Libya MARBERT system.

The final system was not obtained in one training step. It was built in **two main training stages**, preceded by an IDRISI data-cleaning stage:

### Stage 0 — Prepare IDRISI-RA
Raw IDRISI-RA Gold annotations  
→ schema audit and cleaning  
→ flat type-less location dataset (`LOC`)

### Stage 1 — General Arabic Location Training
Base `UBC-NLP/MARBERT`  
→ fine-tune on cleaned IDRISI-RA  
→ Arabic Location NER model (`O / B-LOC / I-LOC`)

### Stage 2 — Libya-specific Balaagh Training
The IDRISI-trained MARBERT is reused as the starting point:

- **Location NER:** continue fine-tuning the complete IDRISI NER model on Balaagh.
- **Incident Category:** reuse the IDRISI-trained MARBERT encoder and initialize a new 6-class sequence-classification head.
- **Priority:** reuse the same IDRISI-trained MARBERT encoder and initialize a new 4-class sequence-classification head.

Therefore, the project has a **common MARBERT backbone lineage**, but the three final task heads are trained as task-specific models in Stage 2.

## 1. Overview

CrisisLens Libya processes Arabic and Libyan-dialect crisis reports and produces three main AI outputs:

1. **Location extraction** — identify explicit location mentions in the report.
2. **Incident category** — classify the report into one of six crisis categories.
3. **Priority** — classify the urgency/severity into Low, Medium, High, or Critical.

The training strategy deliberately begins with a general Arabic location-recognition task before adapting the model to Libya-specific reports.

### Why two training stages?

IDRISI provides a comparatively larger Arabic location-mention dataset. Training MARBERT on it first gives the encoder and NER head general experience recognizing Arabic place mentions. Balaagh then adapts that knowledge to Libyan wording, local locations, and the two additional CrisisLens classification tasks.

### Important architecture clarification

This notebook describes the **two-stage / three-task pipeline actually used in the separate-model experiment**:

- Location NER final model
- Incident-category final model
- Priority final model

They share the same MARBERT training lineage, but they are **three saved task models**, not one three-head unified network. The later unified multi-task experiment is a separate experiment and is not mixed with the results reported here.

> **Output provenance:** The visible outputs in this submission notebook were copied from the already-executed IDRISI training notebook and the already-executed Balaagh three-task notebook. The earlier failed indentation cell from the source notebook was not copied; only the successful corrected evaluation output was retained. IDRISI cleaning V3 did not have stored execution outputs, so its verified final statistics are documented in the markdown summary instead of fabricated as cell outputs.

## 2. Dataset

Two datasets are used sequentially.

### 2.1 IDRISI-RA — Stage 1

The final cleaning pipeline selected the Arabic **Gold random split** only and did not mix it with the time-based split or Silver data.

Final cleaned type-less dataset:

| Statistic | Value |
|---|---:|
| Final tweets | 3,696 |
| Train tweets | 3,234 |
| Dev / Validation tweets | 462 |
| Type-less location mentions | 3,932 |
| Train location mentions | 3,449 |
| Dev location mentions | 483 |
| Invalid offsets after final cleaning | 0 |
| Unmapped labels after final cleaning | 0 |

The model is trained type-less: all valid geographic mentions use one entity type, `LOC`.

### 2.2 Balaagh — Stage 2

The Libya-specific Balaagh dataset contains report text plus:

- `incident_type`
- `priority`
- `location_json`
- `source`
- predefined `train / val / test` splits

The held-out test set used for the reported evaluation contains **236 reports**, including **62 reports marked as real**.

### Labels

**Incident Category — 6 classes**
- Fire/Explosion
- Flood/Severe Weather
- Infrastructure/Utilities
- Road/Transportation
- People at Risk/Medical
- Other

**Priority — 4 classes**
- Low
- Medium
- High
- Critical

**Location NER**
- O
- B-LOC
- I-LOC

## 3. Data Preprocessing

### Phase A — IDRISI schema audit and cleaning

The first IDRISI cleaning attempt revealed a schema mismatch: the public Gold files use fields such as `startIdx / endIdx` and abbreviated location types such as `CITY`, `CTRY`, `HPOI`, and `ST`.

The early cleaner expected different field names, which produced invalid offsets and unmapped labels. The cleaner was therefore revised before any training data was accepted.

The **final cleaning version** used below:

- detects the actual public IDRISI schema;
- maps IDRISI type abbreviations to canonical names;
- maps `ST` to `ROAD_STREET`;
- validates every character span against the original text;
- removes exact duplicate annotations;
- preserves the full audit version with nested/overlapping mentions;
- creates a flat sequence-labeling version using the **longest-span overlap policy**;
- removes only the three same-span type conflicts from the typed version;
- keeps all valid spans for the type-less `LOC` version;
- checks train/dev leakage;
- does not fabricate a public test split;
- exports `idrisi_gold_ner_typeless.jsonl`, which is the file used for Stage 1.

Final cleaning QA:
- 158 exact duplicate mentions removed.
- 51 nested/overlapping cases detected.
- 3 same-span type conflicts detected.
- 3,932 type-less flat mentions retained.
- 0 invalid offsets and 0 unmapped type labels in the final V3 output.

### 3.1 IDRISI cleaning — setup and source discovery

In [ ]:

from pathlib import Path
import subprocess
import json
import gzip
import hashlib
import re
import os
from collections import Counter, defaultdict

import pandas as pd
from IPython.display import display, FileLink

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 120)

# ==============================
# CONFIG
# ==============================
REPO_URL = "https://github.com/rsuwaileh/IDRISI.git"
REPO_DIR = Path("/kaggle/working/IDRISI")
LMR_DIR = REPO_DIR / "LMR"
OUTPUT_DIR = Path("/kaggle/working/idrisi_clean")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# لو تعطل الإنترنت في Kaggle:
# ارفع الداتا يدويًا كـ Kaggle Dataset وضع مسارها هنا.
FALLBACK_SCAN_ROOT = Path("/kaggle/input")

# Default behavior
AUTO_EXCLUDE_SILVER = True
MAX_AUTO_FILE_MB = 250
# IDRISI paper/repo provides alternative partition setups. Do not mix them.
# Default to the random Gold split used as one official benchmark setup.
PREFERRED_PARTITION_MARKER = "gold-random-json"
TRY_UNIQUE_OFFSET_REPAIR = False   # Conservative default: لا نصلح offsets تلقائيًا
REMOVE_CROSS_SPLIT_LEAKAGE = True
OVERLAP_POLICY = "longest"         # longest = الأفضل لـ flat token classification

# إذا أردت تحديد الملفات يدويًا بعد خلية الاكتشاف، ضع المسارات هنا.
# مثال:
# SELECTED_FILES_OVERRIDE = [
#     "/kaggle/working/IDRISI/LMR/.../train.jsonl",
#     "/kaggle/working/IDRISI/LMR/.../dev.jsonl",
#     "/kaggle/working/IDRISI/LMR/.../test.jsonl",
# ]
SELECTED_FILES_OVERRIDE = []

print("Output directory:", OUTPUT_DIR)


In [ ]:

def run_cmd(cmd):
    print("$", " ".join(map(str, cmd)))
    return subprocess.run(cmd, check=True, text=True, capture_output=True)

clone_ok = False

try:
    if (REPO_DIR / ".git").exists():
        print("Repository already exists. Pulling latest changes...")
        result = run_cmd(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
        print(result.stdout[-1500:])
    else:
        result = run_cmd(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
        print(result.stdout[-1500:] if result.stdout else "Clone completed.")
    clone_ok = True
except Exception as e:
    print("GitHub clone/pull failed.")
    print("Reason:", e)
    print("\nFallback: scanning /kaggle/input for manually uploaded files.")

SCAN_ROOT = LMR_DIR if clone_ok and LMR_DIR.exists() else FALLBACK_SCAN_ROOT
print("\nScanning:", SCAN_ROOT)


In [ ]:

AR_RE = re.compile(r"[\u0600-\u06FF]")

def open_text(path):
    path = Path(path)
    if path.suffix == ".gz":
        return gzip.open(path, "rt", encoding="utf-8")
    return open(path, "r", encoding="utf-8")

def read_first_record(path):
    path = Path(path)
    try:
        with open_text(path) as f:
            if path.name.endswith(".json") or path.name.endswith(".json.gz"):
                # Could be JSONL despite .json; try first non-empty line first.
                first = next((line.strip() for line in f if line.strip()), "")
                if not first:
                    return None
                try:
                    obj = json.loads(first)
                    if isinstance(obj, dict):
                        return obj
                    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
                        return obj[0]
                except json.JSONDecodeError:
                    return None
            else:
                for line in f:
                    if line.strip():
                        return json.loads(line)
    except Exception:
        return None
    return None

def arabic_ratio(text):
    if not isinstance(text, str) or not text:
        return 0.0
    nonspace = [c for c in text if not c.isspace()]
    if not nonspace:
        return 0.0
    return sum(bool(AR_RE.match(c)) for c in nonspace) / len(nonspace)

def infer_split(path):
    s = str(path).lower().replace("\\", "/")
    name = Path(path).name.lower()
    if re.search(r"(^|[._/\-])train([._/\-]|$)", s):
        return "train"
    if re.search(r"(^|[._/\-])(dev|valid|validation)([._/\-]|$)", s):
        return "dev"
    if re.search(r"(^|[._/\-])test([._/\-]|$)", s):
        return "test"
    return "unknown"

def infer_event(path):
    parts = [p for p in Path(path).parts if p]
    ignore = {
        "lmr","idrisi","gold","silver","arabic","english",
        "train","dev","test","valid","validation","data","dataset","datasets"
    }
    cleaned = []
    for p in parts[:-1]:
        q = re.sub(r"[^a-z0-9]+", "", p.lower())
        if q and q not in ignore and not q.startswith("kaggle") and q not in {"working","input"}:
            cleaned.append(p)
    return cleaned[-1] if cleaned else "unknown"

json_files = []
for pattern in ("*.jsonl", "*.jsonl.gz", "*.json", "*.json.gz"):
    json_files.extend(SCAN_ROOT.rglob(pattern))

catalog_rows = []
for path in sorted(set(json_files)):
    rec = read_first_record(path)
    if not isinstance(rec, dict):
        continue
    text = rec.get("text", "")
    pstr = str(path).lower()
    size_mb = path.stat().st_size / (1024**2)
    catalog_rows.append({
        "path": str(path),
        "size_mb": round(size_mb, 2),
        "split_guess": infer_split(path),
        "event_guess": infer_event(path),
        "arabic_ratio_first_text": round(arabic_ratio(text), 3),
        "has_location_mentions": isinstance(rec.get("location_mentions"), list),
        "likely_silver": "silver" in pstr,
        "first_keys": ", ".join(rec.keys()),
    })

catalog = pd.DataFrame(catalog_rows)
if catalog.empty:
    raise RuntimeError(
        "No readable JSON/JSONL files found. "
        "Enable Kaggle Internet or upload the IDRISI files manually."
    )

display(catalog.sort_values(
    ["has_location_mentions", "arabic_ratio_first_text", "size_mb"],
    ascending=[False, False, True]
).reset_index(drop=True))


In [ ]:
if SELECTED_FILES_OVERRIDE:
    selected_files = [Path(p) for p in SELECTED_FILES_OVERRIDE]
else:
    candidates = catalog[
        (catalog["has_location_mentions"] == True) &
        (catalog["arabic_ratio_first_text"] >= 0.20)
    ].copy()

    if AUTO_EXCLUDE_SILVER:
        candidates = candidates[candidates["likely_silver"] == False]

    candidates = candidates[candidates["size_mb"] <= MAX_AUTO_FILE_MB]

    # IMPORTANT: random and time-based Gold are alternative partitions
    # of the same underlying Arabic dataset. Never combine them by default.
    preferred = candidates[
        candidates["path"].str.lower().str.contains(
            PREFERRED_PARTITION_MARKER.lower(), regex=False
        )
    ].copy()

    if not preferred.empty:
        print(f"Using preferred partition setup: {PREFERRED_PARTITION_MARKER}")
        candidates = preferred
    else:
        print(
            f"WARNING: '{PREFERRED_PARTITION_MARKER}' was not found. "
            "Using all Arabic non-silver candidates. Review this list carefully."
        )

    selected_files = [Path(p) for p in candidates["path"].tolist()]

if not selected_files:
    raise RuntimeError(
        "Automatic selection found no files. "
        "Inspect the catalog and set SELECTED_FILES_OVERRIDE manually."
    )

print(f"Selected {len(selected_files)} files:\n")
for p in selected_files:
    print(f"[{infer_split(p):7s}] {p}")

partition_families = sorted({
    "gold-random-json" if "gold-random-json" in str(p).lower()
    else "gold-time-json" if "gold-time-json" in str(p).lower()
    else "other"
    for p in selected_files
})

print("\nPartition family/families:", partition_families)
if len(partition_families) > 1:
    print("WARNING: Multiple partition families selected. Consider overriding manually.")

print("\nIMPORTANT: If these include unwanted files, stop here and set SELECTED_FILES_OVERRIDE.")


### 3.2 Load records and inspect the real annotation schema

In [ ]:

def iter_json_records(path):
    path = Path(path)
    with open_text(path) as f:
        # JSONL is the expected format.
        # For .json, first try parsing the whole file; otherwise fall back to JSONL.
        if path.name.endswith(".json") or path.name.endswith(".json.gz"):
            raw = f.read()
            try:
                obj = json.loads(raw)
                if isinstance(obj, list):
                    for x in obj:
                        if isinstance(x, dict):
                            yield x
                    return
                if isinstance(obj, dict):
                    yield obj
                    return
            except json.JSONDecodeError:
                pass

            for line in raw.splitlines():
                if line.strip():
                    yield json.loads(line)
        else:
            for line_no, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as e:
                    raise ValueError(f"Invalid JSON in {path}, line {line_no}: {e}")

raw_records = []

for path in selected_files:
    split = infer_split(path)
    event = infer_event(path)
    count = 0
    for rec in iter_json_records(path):
        if not isinstance(rec, dict):
            continue
        rec = dict(rec)
        rec["_source_file"] = str(path)
        rec["_source_split"] = split
        rec["_source_event"] = event
        raw_records.append(rec)
        count += 1
    print(f"{path.name}: {count:,} records")

print(f"\nTotal loaded records: {len(raw_records):,}")

raw_summary = pd.DataFrame([{
    "split": r.get("_source_split", "unknown"),
    "event": r.get("_source_event", "unknown"),
    "has_text": isinstance(r.get("text"), str),
    "location_mentions": len(r.get("location_mentions") or []),
} for r in raw_records])

display(
    raw_summary.groupby(["split", "event"], dropna=False)
    .agg(tweets=("has_text", "size"), mentions=("location_mentions", "sum"))
    .reset_index()
    .sort_values(["split", "event"])
)


In [ ]:

raw_type_counts = Counter()
mention_key_counts = Counter()

for r in raw_records:
    for m in r.get("location_mentions") or []:
        if isinstance(m, dict):
            raw_type_counts[str(m.get("type"))] += 1
            for k in m.keys():
                mention_key_counts[k] += 1

print("Location mention keys found:")
display(pd.DataFrame(
    mention_key_counts.most_common(),
    columns=["key", "occurrences"]
))

print("\nRaw location type labels found:")
display(pd.DataFrame(
    raw_type_counts.most_common(),
    columns=["raw_type", "count"]
))

has_startidx = any("startIdx" in (m or {}) for r in raw_records for m in (r.get("location_mentions") or []) if isinstance(m, dict))
has_startoffset = any("start_offset" in (m or {}) for r in raw_records for m in (r.get("location_mentions") or []) if isinstance(m, dict))

print("\nOffset schema:")
print("startIdx/endIdx:", has_startidx)
print("start_offset/end_offset:", has_startoffset)


### 3.3 Normalize types and character spans

In [ ]:

TYPE_MAP = {
    # -------- Current public IDRISI Gold abbreviations --------
    "ctry": "COUNTRY",
    "prov": "PROVINCE",
    "city": "CITY_TOWN",
    "dist": "DISTRICT",
    "hpoi": "HUMAN_MADE_POI",
    "npoi": "NATURAL_POI",
    "othr": "OTHER_LOCATION",
    "st": "ROAD_STREET",

    # Defensive aliases for any fine-grained labels that may occur
    "nbhd": "NEIGHBORHOOD",
    "nbrhd": "NEIGHBORHOOD",
    "neigh": "NEIGHBORHOOD",
    "road": "ROAD_STREET",
    "str": "ROAD_STREET",
    "street": "ROAD_STREET",

    # -------- Expanded / human-readable label variants --------
    "country": "COUNTRY",
    "province": "PROVINCE",
    "state": "PROVINCE",
    "state/province": "PROVINCE",
    "province/state": "PROVINCE",
    "city/town": "CITY_TOWN",
    "town": "CITY_TOWN",
    "district": "DISTRICT",
    "neighborhood": "NEIGHBORHOOD",
    "neighbourhood": "NEIGHBORHOOD",
    "road/street": "ROAD_STREET",
    "street/road": "ROAD_STREET",
    "natural point-of-interest": "NATURAL_POI",
    "natural point of interest": "NATURAL_POI",
    "natural poi": "NATURAL_POI",
    "human-made point-of-interest": "HUMAN_MADE_POI",
    "human made point-of-interest": "HUMAN_MADE_POI",
    "human-made point of interest": "HUMAN_MADE_POI",
    "human made point of interest": "HUMAN_MADE_POI",
    "human-made poi": "HUMAN_MADE_POI",
    "human made poi": "HUMAN_MADE_POI",
    "other location": "OTHER_LOCATION",
    "other": "OTHER_LOCATION",
}

def normalize_type_label(label):
    if label is None:
        return "UNMAPPED"
    key = re.sub(r"\s+", " ", str(label).strip().lower())
    return TYPE_MAP.get(key, "UNMAPPED")

def rec_id(rec):
    value = rec.get("tweet_id", rec.get("id", ""))
    return str(value) if value is not None else ""

def text_hash(text):
    return hashlib.sha1((text or "").encode("utf-8")).hexdigest()

def unique_find(text, needle):
    starts = []
    pos = 0
    while True:
        i = text.find(needle, pos)
        if i < 0:
            break
        starts.append(i)
        pos = i + max(1, len(needle))
    return starts

def spans_overlap(a, b):
    return max(a["start_offset"], b["start_offset"]) < min(a["end_offset"], b["end_offset"])

def clean_record(rec, try_repair=False):
    text = rec.get("text")
    rid = rec_id(rec)
    split = rec.get("_source_split", "unknown")
    event = rec.get("_source_event", "unknown")
    issues = []

    if not isinstance(text, str) or not text.strip():
        issues.append({
            "tweet_id": rid, "split": split, "event": event,
            "issue": "missing_or_empty_text", "details": ""
        })
        return None, issues, []

    cleaned_mentions = []
    osm_rows = []
    seen_exact = set()

    for idx, m in enumerate(rec.get("location_mentions") or []):
        if not isinstance(m, dict):
            issues.append({
                "tweet_id": rid, "split": split, "event": event,
                "issue": "mention_not_dict", "details": f"index={idx}"
            })
            continue

        surface = m.get("text")
        # Public IDRISI Gold uses startIdx/endIdx.
        # Some expanded exports use start_offset/end_offset.
        start = m.get("start_offset", m.get("startIdx"))
        end = m.get("end_offset", m.get("endIdx"))
        original_type = m.get("type")
        canonical_type = normalize_type_label(original_type)

        if canonical_type == "UNMAPPED":
            issues.append({
                "tweet_id": rid, "split": split, "event": event,
                "issue": "unmapped_type_label",
                "details": f"index={idx}; type={original_type!r}; text={surface!r}"
            })

        valid_ints = isinstance(start, int) and isinstance(end, int)
        valid_bounds = valid_ints and 0 <= start < end <= len(text)
        surface_match = (
            valid_bounds and isinstance(surface, str) and text[start:end] == surface
        )

        repaired = False
        if (not surface_match) and try_repair and isinstance(surface, str) and surface:
            hits = unique_find(text, surface)
            if len(hits) == 1:
                start = hits[0]
                end = start + len(surface)
                valid_bounds = True
                surface_match = True
                repaired = True
                issues.append({
                    "tweet_id": rid, "split": split, "event": event,
                    "issue": "offset_repaired_unique_exact_match",
                    "details": f"index={idx}; text={surface!r}; new=({start},{end})"
                })

        if not valid_bounds:
            issues.append({
                "tweet_id": rid, "split": split, "event": event,
                "issue": "invalid_offset_bounds",
                "details": f"index={idx}; start={start}; end={end}; text_len={len(text)}; mention={surface!r}"
            })
            continue

        if not isinstance(surface, str) or not surface:
            issues.append({
                "tweet_id": rid, "split": split, "event": event,
                "issue": "empty_mention_text",
                "details": f"index={idx}"
            })
            continue

        if not surface_match:
            issues.append({
                "tweet_id": rid, "split": split, "event": event,
                "issue": "offset_surface_mismatch",
                "details": (
                    f"index={idx}; annotation={surface!r}; "
                    f"slice={text[start:end]!r}; start={start}; end={end}"
                )
            })
            continue

        key = (start, end, surface, str(original_type))
        if key in seen_exact:
            issues.append({
                "tweet_id": rid, "split": split, "event": event,
                "issue": "duplicate_exact_mention_removed",
                "details": f"index={idx}; mention={surface!r}; span=({start},{end}); type={original_type!r}"
            })
            continue
        seen_exact.add(key)

        cm = {
            "text": surface,
            "type": canonical_type,
            "original_type": original_type,
            "start_offset": start,
            "end_offset": end,
        }
        if m.get("keyword") is not None:
            cm["keyword"] = m.get("keyword")
        cleaned_mentions.append(cm)

        # OSM goes to a separate reference table; not into NER features.
        osm = m.get("osm_json") if isinstance(m.get("osm_json"), dict) else {}
        osm_rows.append({
            "tweet_id": rid,
            "mention_text": surface,
            "keyword": m.get("keyword"),
            "canonical_type": canonical_type,
            "original_type": original_type,
            "start_offset": start,
            "end_offset": end,
            "confidence": m.get("confidence"),
            "osm_url": m.get("osm_url"),
            "nominatim_url": m.get("nominatim_url"),
            "osm_id": osm.get("osm_id"),
            "osm_type": osm.get("osm_type"),
            "lat": osm.get("lat"),
            "lon": osm.get("lon"),
            "display_name": osm.get("display_name"),
            "osm_category": osm.get("category"),
            "osm_feature_type": osm.get("type"),
        })

    cleaned_mentions.sort(key=lambda x: (x["start_offset"], x["end_offset"]))

    # Conflicting labels on the same exact span.
    by_span = defaultdict(set)
    for m in cleaned_mentions:
        by_span[(m["start_offset"], m["end_offset"], m["text"])].add(m["type"])
    for span, labels in by_span.items():
        if len(labels) > 1:
            issues.append({
                "tweet_id": rid, "split": split, "event": event,
                "issue": "same_span_conflicting_types",
                "details": f"span={span}; labels={sorted(labels)}"
            })

    # Detect overlaps/nesting, but preserve them in full clean.
    for i in range(len(cleaned_mentions)):
        for j in range(i + 1, len(cleaned_mentions)):
            a, b = cleaned_mentions[i], cleaned_mentions[j]
            if b["start_offset"] >= a["end_offset"]:
                break
            if spans_overlap(a, b):
                issues.append({
                    "tweet_id": rid, "split": split, "event": event,
                    "issue": "overlapping_or_nested_mentions",
                    "details": (
                        f"A={a['text']!r}@({a['start_offset']},{a['end_offset']}); "
                        f"B={b['text']!r}@({b['start_offset']},{b['end_offset']})"
                    )
                })

    clean = {
        "tweet_id": rid,
        "text": text,
        "split": split,
        "event": event,
        "info_class": rec.get("info_class"),
        "location_mentions": cleaned_mentions,
        "_source_file": rec.get("_source_file"),
        "_text_hash": text_hash(text),
    }
    return clean, issues, osm_rows

clean_records = []
qa_issues = []
osm_reference_rows = []

for rec in raw_records:
    clean, issues, osm_rows = clean_record(
        rec, try_repair=TRY_UNIQUE_OFFSET_REPAIR
    )
    qa_issues.extend(issues)
    osm_reference_rows.extend(osm_rows)
    if clean is not None:
        clean_records.append(clean)

print("Cleanable tweets:", f"{len(clean_records):,}")
print("QA issues:", f"{len(qa_issues):,}")
print("OSM reference rows:", f"{len(osm_reference_rows):,}")

issues_df = pd.DataFrame(qa_issues)
if not issues_df.empty:
    display(
        issues_df["issue"].value_counts()
        .rename_axis("issue").reset_index(name="count")
    )


### 3.4 Deduplicate, check leakage, and flatten overlaps

In [ ]:

def merge_mentions_union(mentions_a, mentions_b):
    out = []
    seen = set()
    for m in list(mentions_a) + list(mentions_b):
        key = (
            m.get("start_offset"), m.get("end_offset"),
            m.get("text"), m.get("type"), m.get("original_type")
        )
        if key not in seen:
            seen.add(key)
            out.append(m)
    return sorted(out, key=lambda x: (x["start_offset"], x["end_offset"]))

def merge_within_split(records):
    merged = {}
    local_issues = []
    for r in records:
        stable_id = r["tweet_id"] if r["tweet_id"] else r["_text_hash"]
        key = (r["split"], stable_id)
        if key not in merged:
            merged[key] = dict(r)
        else:
            prev = merged[key]
            if prev["text"] != r["text"]:
                local_issues.append({
                    "tweet_id": r["tweet_id"],
                    "split": r["split"],
                    "event": r["event"],
                    "issue": "same_id_different_text_within_split",
                    "details": f"source={r['_source_file']}"
                })
                # Keep first record conservatively.
            else:
                prev["location_mentions"] = merge_mentions_union(
                    prev["location_mentions"], r["location_mentions"]
                )
                # Preserve event provenance if different.
                events = set(str(prev["event"]).split(" | "))
                events.add(str(r["event"]))
                prev["event"] = " | ".join(sorted(events))
                local_issues.append({
                    "tweet_id": r["tweet_id"],
                    "split": r["split"],
                    "event": r["event"],
                    "issue": "duplicate_tweet_within_split_merged",
                    "details": f"source={r['_source_file']}"
                })
    return list(merged.values()), local_issues

merged_records, merge_issues = merge_within_split(clean_records)
qa_issues.extend(merge_issues)

# Detect same tweet/text across multiple splits.
id_to_splits = defaultdict(set)
hash_to_splits = defaultdict(set)
for r in merged_records:
    if r["tweet_id"]:
        id_to_splits[r["tweet_id"]].add(r["split"])
    hash_to_splits[r["_text_hash"]].add(r["split"])

cross_ids = {k: v for k, v in id_to_splits.items() if len(v - {"unknown"}) > 1}
cross_hashes = {k: v for k, v in hash_to_splits.items() if len(v - {"unknown"}) > 1}

print("Cross-split duplicate Tweet IDs:", len(cross_ids))
print("Cross-split duplicate text hashes:", len(cross_hashes))

for tid, splits in cross_ids.items():
    qa_issues.append({
        "tweet_id": tid,
        "split": " | ".join(sorted(splits)),
        "event": "",
        "issue": "cross_split_duplicate_tweet_id",
        "details": f"splits={sorted(splits)}"
    })

SPLIT_PRIORITY = {"test": 3, "dev": 2, "train": 1, "unknown": 0}

def remove_cross_split_duplicates(records):
    grouped = defaultdict(list)
    for r in records:
        stable_id = r["tweet_id"] if r["tweet_id"] else r["_text_hash"]
        grouped[stable_id].append(r)

    kept = []
    removed = []
    for stable_id, group in grouped.items():
        if len(group) == 1:
            kept.append(group[0])
            continue
        best = max(group, key=lambda r: SPLIT_PRIORITY.get(r["split"], 0))
        kept.append(best)
        for r in group:
            if r is not best:
                removed.append(r)
    return kept, removed

if REMOVE_CROSS_SPLIT_LEAKAGE:
    dedup_records, removed_leakage = remove_cross_split_duplicates(merged_records)
    print("Removed lower-priority cross-split copies:", len(removed_leakage))
    for r in removed_leakage:
        qa_issues.append({
            "tweet_id": r["tweet_id"],
            "split": r["split"],
            "event": r["event"],
            "issue": "removed_to_prevent_cross_split_leakage",
            "details": r["_source_file"]
        })
else:
    dedup_records = merged_records

print("Tweets after split-safe dedup:", f"{len(dedup_records):,}")


In [ ]:

def remove_same_span_type_conflicts(mentions):
    """
    For typed NER only:
    if the exact same text span has >1 different type label,
    exclude that span from the typed flat training set rather than choosing
    one label arbitrarily. The full-clean audit file still keeps the originals.
    """
    span_to_types = defaultdict(set)
    for m in mentions:
        key = (m["start_offset"], m["end_offset"], m["text"])
        span_to_types[key].add(m["type"])

    conflicting_keys = {
        key for key, labels in span_to_types.items()
        if len(labels) > 1
    }

    kept = [
        m for m in mentions
        if (m["start_offset"], m["end_offset"], m["text"]) not in conflicting_keys
    ]
    return kept, conflicting_keys


def flatten_mentions_longest(mentions):
    if not mentions:
        return []

    ranked = sorted(
        mentions,
        key=lambda m: (
            -(m["end_offset"] - m["start_offset"]),
            m["start_offset"],
            m["end_offset"]
        )
    )

    chosen = []
    for m in ranked:
        if not any(spans_overlap(m, x) for x in chosen):
            chosen.append(m)

    return sorted(chosen, key=lambda m: (m["start_offset"], m["end_offset"]))

ner_flat_records = []          # typed
ner_typeless_records = []      # primary recommendation for CrisisLens location extraction
flat_dropped_count = 0
typed_conflict_spans_dropped = 0

for r in dedup_records:
    # First remove exact-span conflicting type labels ONLY for typed NER.
    typed_source_mentions, conflicting_keys = remove_same_span_type_conflicts(
        r["location_mentions"]
    )
    typed_conflict_spans_dropped += len(conflicting_keys)

    # Typed flat entities
    flat_mentions = flatten_mentions_longest(typed_source_mentions)
    flat_dropped_count += len(typed_source_mentions) - len(flat_mentions)

    typed_entities = [{
        "start": m["start_offset"],
        "end": m["end_offset"],
        "text": m["text"],
        "label": m["type"],
        "original_type": m["original_type"],
    } for m in flat_mentions]

    ner_flat_records.append({
        "tweet_id": r["tweet_id"],
        "text": r["text"],
        "split": r["split"],
        "event": r["event"],
        "info_class": r["info_class"],
        "entities": typed_entities,
    })

    # Type-less flat entities:
    # every valid annotated location becomes LOC.
    # This is the recommended first model for CrisisLens because the immediate
    # goal is to detect a place mention, then resolve it with OSM/Nominatim.
    typeless_mentions = flatten_mentions_longest(r["location_mentions"])

    typeless_entities = [{
        "start": m["start_offset"],
        "end": m["end_offset"],
        "text": m["text"],
        "label": "LOC",
    } for m in typeless_mentions]

    ner_typeless_records.append({
        "tweet_id": r["tweet_id"],
        "text": r["text"],
        "split": r["split"],
        "event": r["event"],
        "info_class": r["info_class"],
        "entities": typeless_entities,
    })

print("Typed flat NER tweets:", f"{len(ner_flat_records):,}")
print("Type-less flat NER tweets:", f"{len(ner_typeless_records):,}")
print("Typed exact-span conflicts excluded:", f"{typed_conflict_spans_dropped:,}")
print("Typed mentions omitted due to overlap:", f"{flat_dropped_count:,}")
print("Type-less mentions:", f"{sum(len(r['entities']) for r in ner_typeless_records):,}")


### 3.5 Cleaning QA and export

In [ ]:

summary_rows = []
type_counter = Counter()

for r in ner_flat_records:
    summary_rows.append({
        "split": r["split"],
        "event": r["event"],
        "num_entities": len(r["entities"]),
        "has_location": len(r["entities"]) > 0,
    })
    for e in r["entities"]:
        type_counter[e["label"]] += 1

summary_df = pd.DataFrame(summary_rows)

print("=== Tweets / mentions by split ===")
display(
    summary_df.groupby("split", dropna=False)
    .agg(
        tweets=("num_entities", "size"),
        mentions=("num_entities", "sum"),
        tweets_without_location=("has_location", lambda x: int((~x).sum()))
    )
    .reset_index()
)

print("\n=== Canonical location types in flat NER ===")
display(pd.DataFrame(type_counter.most_common(), columns=["type", "count"]))

issues_df = pd.DataFrame(qa_issues)
print("\n=== QA issues ===")
if issues_df.empty:
    print("No QA issues recorded.")
else:
    display(
        issues_df["issue"].value_counts()
        .rename_axis("issue").reset_index(name="count")
    )

print("\n=== Sample cleaned records ===")
display(pd.DataFrame([{
    "tweet_id": r["tweet_id"],
    "split": r["split"],
    "text": r["text"],
    "entities": r["entities"],
} for r in ner_flat_records[:5]]))


available_splits = sorted(set(r["split"] for r in ner_flat_records))
print("\n=== Split availability ===")
print("Available splits:", available_splits)
if "test" not in available_splits:
    print(
        "NOTE: No test.jsonl was found in the selected public gold-random-json files. "
        "Do not manufacture a test set silently. For model development, keep the official "
        "train/dev split, and later use a separate held-out Libya-specific test set."
    )


print("\n=== CrisisLens recommended type-less dataset ===")
typeless_counts = pd.DataFrame([{
    "split": r["split"],
    "num_entities": len(r["entities"]),
} for r in ner_typeless_records])

display(
    typeless_counts.groupby("split", dropna=False)
    .agg(
        tweets=("num_entities", "size"),
        location_mentions=("num_entities", "sum")
    )
    .reset_index()
)


In [ ]:

def write_jsonl(path, records):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Full clean: strip internal fields before export.
full_export = []
for r in dedup_records:
    full_export.append({
        "tweet_id": r["tweet_id"],
        "text": r["text"],
        "split": r["split"],
        "event": r["event"],
        "info_class": r["info_class"],
        "location_mentions": r["location_mentions"],
    })

full_path = OUTPUT_DIR / "idrisi_gold_clean_full.jsonl"
flat_path = OUTPUT_DIR / "idrisi_gold_ner_typed.jsonl"
typeless_path = OUTPUT_DIR / "idrisi_gold_ner_typeless.jsonl"
mentions_csv_path = OUTPUT_DIR / "idrisi_mentions_flat.csv"
qa_path = OUTPUT_DIR / "idrisi_qa_issues.csv"
osm_path = OUTPUT_DIR / "idrisi_osm_reference.csv"
summary_path = OUTPUT_DIR / "idrisi_cleaning_summary.json"

write_jsonl(full_path, full_export)
write_jsonl(flat_path, ner_flat_records)
write_jsonl(typeless_path, ner_typeless_records)

# Expanded flat mentions CSV
mention_rows = []
for r in ner_flat_records:
    if not r["entities"]:
        mention_rows.append({
            "tweet_id": r["tweet_id"],
            "split": r["split"],
            "event": r["event"],
            "text": r["text"],
            "mention_text": None,
            "start": None,
            "end": None,
            "label": None,
            "original_type": None,
        })
    else:
        for e in r["entities"]:
            mention_rows.append({
                "tweet_id": r["tweet_id"],
                "split": r["split"],
                "event": r["event"],
                "text": r["text"],
                "mention_text": e["text"],
                "start": e["start"],
                "end": e["end"],
                "label": e["label"],
                "original_type": e["original_type"],
            })

pd.DataFrame(mention_rows).to_csv(
    mentions_csv_path, index=False, encoding="utf-8-sig"
)

pd.DataFrame(qa_issues).to_csv(
    qa_path, index=False, encoding="utf-8-sig"
)

osm_df = pd.DataFrame(osm_reference_rows)
if not osm_df.empty:
    # Remove exact OSM reference duplicates only.
    dedup_cols = [
        c for c in ["mention_text", "keyword", "osm_id", "lat", "lon"]
        if c in osm_df.columns
    ]
    osm_df = osm_df.drop_duplicates(subset=dedup_cols)
osm_df.to_csv(osm_path, index=False, encoding="utf-8-sig")

summary = {
    "source_repo": REPO_URL,
    "selected_files": [str(p) for p in selected_files],
    "raw_records": len(raw_records),
    "clean_records_before_split_dedup": len(clean_records),
    "final_tweets": len(ner_flat_records),
    "typed_flat_mentions": sum(len(r["entities"]) for r in ner_flat_records),
    "typeless_flat_mentions": sum(len(r["entities"]) for r in ner_typeless_records),
    "typed_exact_span_conflicts_dropped": typed_conflict_spans_dropped,
    "mentions_dropped_due_to_overlap_for_flat_ner": flat_dropped_count,
    "qa_issue_counts": Counter(x["issue"] for x in qa_issues),
    "settings": {
        "TRY_UNIQUE_OFFSET_REPAIR": TRY_UNIQUE_OFFSET_REPAIR,
        "REMOVE_CROSS_SPLIT_LEAKAGE": REMOVE_CROSS_SPLIT_LEAKAGE,
        "OVERLAP_POLICY": OVERLAP_POLICY,
    },
    "license_note": "IDRISI repository states datasets are licensed under CC BY 4.0. Preserve attribution.",
}

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

# ZIP
import shutil
zip_base = Path("/kaggle/working/idrisi_clean_bundle")
zip_path = shutil.make_archive(
    str(zip_base), "zip",
    root_dir=str(OUTPUT_DIR)
)

print("Saved files:")
for p in [full_path, flat_path, typeless_path, mentions_csv_path, qa_path, osm_path, summary_path]:
    print(" -", p, f"({p.stat().st_size / 1024:.1f} KB)")
print(" -", zip_path)

print("\nDownload bundle:")
display(FileLink(zip_path))


### Phase B — Convert cleaned IDRISI character spans to MARBERT BIO labels

IDRISI annotations are stored as character spans, while MARBERT is trained at token/subtoken level.

A **Fast Tokenizer** is required because `offset_mapping` links every MARBERT token back to its original character positions.

For each explicit location:
- the first overlapping token receives `B-LOC`;
- subsequent overlapping tokens receive `I-LOC`;
- tokens outside locations receive `O`;
- special and padding tokens receive `-100` and are ignored by the loss.

The alignment QA confirmed:

| Split | Gold entities | Aligned entities | Lost by truncation |
|---|---:|---:|---:|
| Train | 3,449 | 3,449 | 0 |
| Dev | 483 | 483 | 0 |

No gold entity was lost because of truncation.

### 3.6 Stage 1 imports and reproducibility

In [2]:
from pathlib import Path
import json
import random
import shutil
from collections import Counter

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    set_seed,
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: فعّل GPU من Kaggle > Settings > Accelerator")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


### 3.7 Stage 1 configuration

In [3]:
MODEL_NAME = "UBC-NLP/MARBERT"
# للمقارنة لاحقًا:
# MODEL_NAME = "UBC-NLP/MARBERTv2"

DATA_FILENAME = "idrisi_gold_ner_typeless.jsonl"

MAX_LENGTH = 128
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10

OUTPUT_DIR = Path("/kaggle/working/marbert_idrisi_typeless")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_LIST = ["O", "B-LOC", "I-LOC"]
label2id = {label: i for i, label in enumerate(LABEL_LIST)}
id2label = {i: label for label, i in label2id.items()}

print("Model:", MODEL_NAME)
print("Labels:", label2id)
print("Output:", OUTPUT_DIR)

Model: UBC-NLP/MARBERT
Labels: {'O': 0, 'B-LOC': 1, 'I-LOC': 2}
Output: /kaggle/working/marbert_idrisi_typeless


### 3.8 Locate the cleaned type-less IDRISI file

In [ ]:
matches = list(Path("/kaggle/working").rglob(DATA_FILENAME))
matches += [p for p in Path("/kaggle/input").rglob(DATA_FILENAME) if p not in matches]

if not matches:
    print("IDRISI-like files found:")
    for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
        for p in root.rglob("*idrisi*"):
            if p.is_file():
                print(" -", p)
    raise FileNotFoundError(
        f"لم أجد {DATA_FILENAME}. تأكد أن Kaggle Dataset مضافة للـNotebook."
    )

for i, p in enumerate(matches):
    print(i, p)

DATA_PATH = matches[0]
print("\nUsing:", DATA_PATH)

### 3.9 Load and validate Stage 1 data

In [5]:
records = []

with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as e:
            raise ValueError(f"JSON error at line {line_no}: {e}")

print("Total records:", len(records))
print("Keys:", records[0].keys())

required = {"text", "split", "entities"}
missing = required - set(records[0].keys())
if missing:
    raise ValueError(f"Missing required fields: {missing}")

display(pd.DataFrame([{
    "tweet_id": r.get("tweet_id"),
    "split": r.get("split"),
    "text": r.get("text"),
    "entities": r.get("entities"),
} for r in records[:5]]))

Total records: 3696
Keys: dict_keys(['tweet_id', 'text', 'split', 'event', 'info_class', 'entities'])


,tweet_id,split,text,entities
0,1290747020844695552,dev,٣٠٠٠ جريح و ٥٠ شهيد إهراءات القمح (مخزون لبنان...,"[{'start': 41, 'end': 46, 'text': 'لبنان', 'la..."
1,1290747381600976897,dev,🔴 وزير الصحة اللبناني: انقطاع التيار الكهربائى...,"[{'start': 68, 'end': 73, 'text': 'بيروت', 'la..."
2,1290999307718594562,dev,فيديو من زاوية قريبة جدًا لحظة #انفجار_المرفأ ...,"[{'start': 49, 'end': 54, 'text': 'بيروت', 'la..."
3,1290694625464852480,dev,١٠ عناصر من فوج بيروت مفقودين!,"[{'start': 16, 'end': 21, 'text': 'بيروت', 'la..."
4,1290999206203854848,dev,الأحداث - مستشفى الحريري تعلن عن عدد ضحايا انف...,"[{'start': 10, 'end': 24, 'text': 'مستشفى الحر..."


In [6]:
qa_errors = []
split_stats = Counter()
entity_stats = Counter()

for row_idx, r in enumerate(records):
    text = r.get("text", "")
    split = str(r.get("split", "")).lower()
    entities = r.get("entities") or []

    split_stats[split] += 1
    entity_stats[split] += len(entities)

    if not isinstance(text, str) or not text:
        qa_errors.append((row_idx, "empty_text", ""))

    prev_end = -1

    for e in entities:
        start = e.get("start")
        end = e.get("end")
        surface = e.get("text")
        label = e.get("label")

        if label != "LOC":
            qa_errors.append((row_idx, "unexpected_label", repr(label)))

        if not isinstance(start, int) or not isinstance(end, int):
            qa_errors.append((row_idx, "non_integer_offset", repr(e)))
            continue

        if not (0 <= start < end <= len(text)):
            qa_errors.append((row_idx, "invalid_bounds", repr(e)))
            continue

        if text[start:end] != surface:
            qa_errors.append((
                row_idx,
                "surface_mismatch",
                f"expected={surface!r}, slice={text[start:end]!r}"
            ))

        if start < prev_end:
            qa_errors.append((row_idx, "overlap_in_flat_dataset", repr(e)))

        prev_end = max(prev_end, end)

print("Tweets by split:", dict(split_stats))
print("Entities by split:", dict(entity_stats))
print("QA errors:", len(qa_errors))

if qa_errors:
    display(pd.DataFrame(qa_errors[:100], columns=["row", "issue", "details"]))
    raise ValueError("QA failed. لا تبدأ التدريب قبل حل المشاكل.")

if "train" not in split_stats or "dev" not in split_stats:
    raise ValueError("نحتاج split=train و split=dev.")

print("✅ Dataset QA passed.")

Tweets by split: {'dev': 462, 'train': 3234}
Entities by split: {'dev': 483, 'train': 3449}
QA errors: 0
✅ Dataset QA passed.


### 3.10 Load MARBERT tokenizer and align character spans to BIO tokens

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

if not tokenizer.is_fast:
    raise RuntimeError(
        "نحتاج Fast Tokenizer لأن alignment يعتمد على offset_mapping."
    )

print("Tokenizer:", tokenizer.__class__.__name__)
print("Vocab size:", tokenizer.vocab_size)
print("Model max length:", tokenizer.model_max_length)

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer: BertTokenizerFast
Vocab size: 100000
Model max length: 1000000000000000019884624838656


In [8]:
def char_spans_to_bio(text, entities, tokenizer, max_length=128):
    enc = tokenizer(
        text,
        truncation=True,
        max_length=max_length,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )

    offsets = enc["offset_mapping"]
    labels = [-100] * len(offsets)

    real_token_indices = []
    for i, (tok_start, tok_end) in enumerate(offsets):
        if tok_start == tok_end:
            labels[i] = -100
        else:
            labels[i] = label2id["O"]
            real_token_indices.append(i)

    aligned_entities = 0
    truncated_entities = 0

    for ent in entities:
        ent_start = int(ent["start"])
        ent_end = int(ent["end"])

        overlapping = []
        for i in real_token_indices:
            tok_start, tok_end = offsets[i]
            if tok_end > ent_start and tok_start < ent_end:
                overlapping.append(i)

        if not overlapping:
            truncated_entities += 1
            continue

        labels[overlapping[0]] = label2id["B-LOC"]
        for i in overlapping[1:]:
            labels[i] = label2id["I-LOC"]

        aligned_entities += 1

    enc.pop("offset_mapping")
    enc["labels"] = labels

    return enc, aligned_entities, truncated_entities


example = next(r for r in records if r.get("entities"))
encoded_example, aligned, truncated = char_spans_to_bio(
    example["text"], example["entities"], tokenizer, MAX_LENGTH
)

tokens = tokenizer.convert_ids_to_tokens(encoded_example["input_ids"])
preview = []
for tok, lab in zip(tokens, encoded_example["labels"]):
    preview.append({
        "token": tok,
        "label": "IGNORED" if lab == -100 else id2label[lab],
    })

print("Text:", example["text"])
print("Entities:", example["entities"])
print("Aligned:", aligned, "| Truncated:", truncated)
display(pd.DataFrame(preview))

Text: ٣٠٠٠ جريح و ٥٠ شهيد إهراءات القمح (مخزون لبنان لـ ٦ أشهر) تدمرت، مستودعات أدوية الأمراض المُستعصية تدمر بجزء كبير منها ، مرفأ بيروت خارج الخدمة، المطار شبه مغلق، شركة الكهرباء شبه مدمرة، ومع تفشي كوروناوهبوط سعر صرف الليرة نحن على أبواب الموت لازم ننتقم #انفجار_بيروت #بيروت
Entities: [{'start': 41, 'end': 46, 'text': 'لبنان', 'label': 'LOC'}, {'start': 65, 'end': 98, 'text': 'مستودعات أدوية الأمراض المُستعصية', 'label': 'LOC'}, {'start': 121, 'end': 131, 'text': 'مرفأ بيروت', 'label': 'LOC'}]
Aligned: 3 | Truncated: 0


,token,label
0,[CLS],IGNORED
1,٣٠٠,O
2,##٠,O
3,جريح,O
4,و,O
...,...,...
61,[UNK],O
62,بيروت,O
63,[UNK],O
64,بيروت,O


In [9]:
train_records = [r for r in records if str(r.get("split")).lower() == "train"]
dev_records = [r for r in records if str(r.get("split")).lower() == "dev"]

print("Train:", len(train_records))
print("Dev:", len(dev_records))

def encode_dataset(rows):
    encoded_rows = []
    total_gold = 0
    aligned_total = 0
    truncated_total = 0

    for r in rows:
        entities = r.get("entities") or []
        total_gold += len(entities)

        enc, aligned, truncated = char_spans_to_bio(
            r["text"], entities, tokenizer, MAX_LENGTH
        )

        encoded_rows.append(enc)
        aligned_total += aligned
        truncated_total += truncated

    return Dataset.from_list(encoded_rows), {
        "gold_entities": total_gold,
        "aligned_entities": aligned_total,
        "truncated_entities": truncated_total,
    }

train_ds, train_alignment = encode_dataset(train_records)
dev_ds, dev_alignment = encode_dataset(dev_records)

print("Train alignment:", train_alignment)
print("Dev alignment:", dev_alignment)

if train_alignment["truncated_entities"] or dev_alignment["truncated_entities"]:
    raise ValueError(
        "بعض locations ضاعت بسبب MAX_LENGTH. ارفع MAX_LENGTH وأعد تشغيل الـNotebook."
    )

print("✅ No gold entity was lost by truncation.")
print(train_ds)
print(dev_ds)

Train: 3234
Dev: 462
Train alignment: {'gold_entities': 3449, 'aligned_entities': 3449, 'truncated_entities': 0}
Dev alignment: {'gold_entities': 483, 'aligned_entities': 483, 'truncated_entities': 0}
✅ No gold entity was lost by truncation.
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3234
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 462
})


In [10]:
def count_token_labels(ds):
    counts = Counter()
    for row in ds:
        for y in row["labels"]:
            if y != -100:
                counts[id2label[y]] += 1
    return counts

train_label_counts = count_token_labels(train_ds)
dev_label_counts = count_token_labels(dev_ds)

print("Train:", train_label_counts)
print("Dev:", dev_label_counts)

if train_label_counts["B-LOC"] == 0:
    raise RuntimeError("B-LOC = 0. يوجد خطأ في alignment.")

display(pd.DataFrame([
    {"split": "train", **dict(train_label_counts)},
    {"split": "dev", **dict(dev_label_counts)},
]).fillna(0))

Train: Counter({'O': 140769, 'B-LOC': 3449, 'I-LOC': 3009})
Dev: Counter({'O': 20481, 'B-LOC': 483, 'I-LOC': 426})


,split,O,B-LOC,I-LOC
0,train,140769,3449,3009
1,dev,20481,483,426


## 4. Model / Method

### Stage 1 — MARBERT Location NER on IDRISI

Base model:

`UBC-NLP/MARBERT`

A token-classification head with three labels is attached:

`O / B-LOC / I-LOC`

This stage teaches general Arabic location-mention recognition.

### Stage 2 — Balaagh adaptation

The saved IDRISI model becomes the starting checkpoint.

#### Location model
`IDRISI NER model → continue NER fine-tuning on Balaagh`

The existing NER head is preserved and updated.

#### Incident-category model
`IDRISI-trained MARBERT encoder → new 6-class classification head`

The old NER head is not reused as a classifier. A new sequence-classification head is initialized, while the MARBERT encoder starts from the IDRISI-trained weights.

#### Priority model
`IDRISI-trained MARBERT encoder → new 4-class classification head`

Again, the encoder is reused and the classification head is newly initialized.

This design lets all three downstream tasks benefit from the same Arabic MARBERT backbone while keeping each task's output layer appropriate to its objective.

### 4.1 Stage 1 token-classification model

In [ ]:

def bio_to_spans(tags):
    """
    Convert BIO sequence to exact entity spans:
    [(start_token, end_token_exclusive, label), ...]
    Robust to stray I-LOC by starting a new entity.
    """
    spans = []
    start = None
    ent_type = None

    for i, tag in enumerate(tags + ["O"]):
        if tag == "O":
            if start is not None:
                spans.append((start, i, ent_type))
                start = None
                ent_type = None
            continue

        if "-" in tag:
            prefix, label = tag.split("-", 1)
        else:
            prefix, label = tag, "LOC"

        if prefix == "B":
            if start is not None:
                spans.append((start, i, ent_type))
            start = i
            ent_type = label

        elif prefix == "I":
            if start is None or ent_type != label:
                if start is not None:
                    spans.append((start, i, ent_type))
                start = i
                ent_type = label

        else:
            if start is not None:
                spans.append((start, i, ent_type))
                start = None
                ent_type = None

    return spans


def entity_metrics(gold_sequences, pred_sequences):
    tp = fp = fn = 0
    correct_tokens = total_tokens = 0

    for gold, pred in zip(gold_sequences, pred_sequences):
        gold_spans = set(bio_to_spans(gold))
        pred_spans = set(bio_to_spans(pred))

        tp += len(gold_spans & pred_spans)
        fp += len(pred_spans - gold_spans)
        fn += len(gold_spans - pred_spans)

        correct_tokens += sum(g == p for g, p in zip(gold, pred))
        total_tokens += len(gold)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) else 0.0
    )
    token_accuracy = correct_tokens / total_tokens if total_tokens else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "token_accuracy": token_accuracy,
        "tp": tp,
        "fp": fp,
        "fn": fn,
    }


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    pred_sequences = []
    gold_sequences = []

    for pred_row, gold_row in zip(preds, labels):
        pred_tags = []
        gold_tags = []

        for p, g in zip(pred_row, gold_row):
            if g == -100:
                continue
            pred_tags.append(id2label[int(p)])
            gold_tags.append(id2label[int(g)])

        pred_sequences.append(pred_tags)
        gold_sequences.append(gold_tags)

    m = entity_metrics(gold_sequences, pred_sequences)

    # Trainer metrics should contain simple scalars only.
    return {
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "token_accuracy": m["token_accuracy"],
    }


In [12]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
)

num_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {num_params/1e6:.1f}M")

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at UBC-NLP/MARBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Parameters: 162.3M


## 5. Training / Processing

## Training Stage 1 — IDRISI Location NER

Configuration used:

| Parameter | Value |
|---|---:|
| Base model | UBC-NLP/MARBERT |
| Max length | 128 tokens |
| Learning rate | 2e-5 |
| Train batch size | 16 |
| Eval batch size | 32 |
| Epochs | 5 |
| Weight decay | 0.01 |
| Warm-up ratio | 0.10 |
| Model selection | Dev entity F1 |
| FP16 | Enabled when GPU is available |
| Seed | 42 |

The best checkpoint is selected only from Dev/Validation F1.

### 5.1 Stage 1 Trainer setup

In [13]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,

    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=25,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,

    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
    data_seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer ready.")

Trainer ready.


### 5.2 Stage 1 training

In [14]:
train_result = trainer.train()

print("Training completed.")
print(train_result)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,0.048100,0.041411,0.671480,0.770186,0.717454,0.985788
2,0.027500,0.030262,0.781132,0.857143,0.817374,0.989294
3,0.016800,0.028656,0.829960,0.848861,0.839304,0.990930
4,0.010700,0.030615,0.807692,0.869565,0.837488,0.990977
5,0.009100,0.033129,0.803883,0.857143,0.829659,0.990977


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

Training completed.
TrainOutput(global_step=510, training_loss=0.08895954965960745, metrics={'train_runtime': 232.9468, 'train_samples_per_second': 69.415, 'train_steps_per_second': 2.189, 'total_flos': 721542813616440.0, 'train_loss': 0.08895954965960745, 'epoch': 5.0})


### Stage 1 Recorded Result

After 5 epochs:

- Dev Precision: **0.8300**
- Dev Recall: **0.8489**
- Dev entity F1: **0.8393**
- Token Accuracy: **0.9909**
- True Positives: **410**
- False Positives: **84**
- False Negatives: **73**

The trained model is saved as `marbert_idrisi_typeless/final_model` and becomes the starting checkpoint for the Balaagh stage.

### 5.3 Save Stage 1 model

In [19]:
FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

metrics_to_save = {}
for k, v in dev_metrics.items():
    if isinstance(v, (float, np.floating)):
        metrics_to_save[k] = float(v)
    else:
        metrics_to_save[k] = v

with open(OUTPUT_DIR / "dev_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_to_save, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump({
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "learning_rate": LEARNING_RATE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "weight_decay": WEIGHT_DECAY,
        "warmup_ratio": WARMUP_RATIO,
        "seed": SEED,
        "labels": LABEL_LIST,
        "train_size": len(train_records),
        "dev_size": len(dev_records),
        "train_alignment": train_alignment,
        "dev_alignment": dev_alignment,
    }, f, ensure_ascii=False, indent=2)

with open(OUTPUT_DIR / "trainer_log_history.json", "w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, ensure_ascii=False, indent=2)

print("Saved model:", FINAL_MODEL_DIR)
print("Saved metrics:", OUTPUT_DIR / "dev_metrics.json")

Saved model: /kaggle/working/marbert_idrisi_typeless/final_model
Saved metrics: /kaggle/working/marbert_idrisi_typeless/dev_metrics.json


## Training Stage 2 — Balaagh Three-Task Adaptation

The second training stage uses the Libya-specific Balaagh dataset.

Configuration:

| Setting | Location NER | Category / Priority |
|---|---:|---:|
| Max length | 192 | 192 |
| Epochs | 4 | 5 |
| Learning rate | 1e-5 | 2e-5 |
| Train batch | 8 | 8 |
| Eval batch | 16 | 16 |
| Gradient accumulation | 2 | 2 |
| Effective batch | 16 | 16 |
| Weight decay | 0.01 | 0.01 |
| Warm-up ratio | 0.10 | 0.10 |
| Early stopping patience | 2 | 2 |
| Best metric | Entity F1 | Macro F1 |
| Seed | 42 | 42 |

The same Balaagh train/validation/test partitions are retained.

### 5.4 Stage 2 configuration

In [ ]:
from pathlib import Path
import os, re, io, csv, json, zipfile, shutil, gc, hashlib, random
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import (AutoTokenizer, AutoConfig, AutoModelForTokenClassification,
                          AutoModelForSequenceClassification, DataCollatorForTokenClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer,
                          EarlyStoppingCallback, set_seed)

SEED = 42
set_seed(SEED); random.seed(SEED); np.random.seed(SEED)
ROOTS = [Path('/kaggle/input'), Path('/kaggle/working')]
OUTPUT = Path('/kaggle/working/crisislens_three_tasks')
OUTPUT.mkdir(parents=True, exist_ok=True)

MANUAL_DATA_PATH = ''    # optional: ZIP or the folder containing train.csv
MANUAL_IDRISI_MODEL_PATH = (
    '/kaggle/working/marbert_idrisi_typeless/final_model'
    if Path('/kaggle/working/marbert_idrisi_typeless/final_model').exists()
    else ''
)   # optional: extracted model directory or IDRISI ZIP
MAX_LENGTH = 192
NER_EPOCHS, CLASS_EPOCHS = 4, 5
NER_LR, CLASS_LR = 1e-5, 2e-5
TRAIN_BATCH, EVAL_BATCH, GRAD_ACCUM = 8, 16, 2
RUN_TRAINING = True       # True = train all three tasks

CATEGORIES = ['Fire/Explosion','Flood/Severe Weather','Infrastructure/Utilities',
              'Road/Transportation','People at Risk/Medical','Other']
PRIORITIES = ['Low','Medium','High','Critical']
NER_LABELS = ['O','B-LOC','I-LOC']
print('CUDA:',torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
if RUN_TRAINING and not torch.cuda.is_available():
    raise RuntimeError('فعّلي GPU من Kaggle → Settings → Accelerator قبل Run All.')

### 5.5 Load Balaagh splits

In [3]:
def all_files(filename):
    return sorted({p for root in ROOTS if root.exists() for p in root.rglob(filename) if p.is_file()},
                  key=lambda x:(str(x).count('/'),str(x)))

def load_zip_splits(path):
    with zipfile.ZipFile(path) as z:
        output={}
        for split in ('train','val','test'):
            choices=[s for s in z.namelist() if s == split+'.csv' or s.endswith('/'+split+'.csv')]
            if len(choices)!=1:raise ValueError(f'{path}: expected exactly one {split}.csv, found {choices}')
            with z.open(choices[0]) as f:
                output[split]=pd.read_csv(io.BytesIO(f.read()),encoding='utf-8-sig',keep_default_na=False)
    return output

def load_folder_splits(folder):
    return {s:pd.read_csv(folder/(s+'.csv'),encoding='utf-8-sig',keep_default_na=False)
            for s in ('train','val','test')}

if MANUAL_DATA_PATH:
    DATA_PATH=Path(MANUAL_DATA_PATH)
else:
    zips=all_files('balaagh_dataset_ner_ready.zip')
    if zips: DATA_PATH=zips[0]
    else:
        folders=[p.parent for p in all_files('train.csv')
                 if (p.parent/'val.csv').exists() and (p.parent/'test.csv').exists()]
        if not folders: raise FileNotFoundError('أضيفي Balaagh (النسخة المصححة) من Add Input.')
        DATA_PATH=folders[0]
SPLITS=load_zip_splits(DATA_PATH) if DATA_PATH.suffix=='.zip' else load_folder_splits(DATA_PATH)

# إصلاح موثق لسجل واحد معروف في النسخة الخام من Balaagh.
# هذه ليست قاعدة إصلاح تلقائي عامة: نتدخل فقط عندما يطابق النص والـoffsets
# والـannotation الخطأ الموثق بالضبط؛ النسخة المصححة تمر دون تعديل.
KNOWN_ANNOTATION_FIXES = []
for idx, row in SPLITS['train'].iterrows():
    report = str(row['report'])
    if report != 'انتشال جثة شاب من شاطئ القويعة طرابلس بعد حادث غرق ادى الى وفاة شخصين':
        continue
    try:
        entities = json.loads(row['location_json'])
    except (TypeError, json.JSONDecodeError):
        continue  # QA below will explain a malformed JSON error
    if (len(entities) == 1
        and entities[0].get('start') == 18
        and entities[0].get('end') == 37
        and entities[0].get('text') == 'شاطئ القويعة في طرابلس'
        and entities[0].get('label') == 'LOC'):
        corrected_text = report[18:37]
        if corrected_text != 'شاطئ القويعة طرابلس':
            raise ValueError('Unexpected text for known fix; review the source record manually.')
        entities[0]['text'] = corrected_text
        SPLITS['train'].at[idx, 'location_json'] = json.dumps(entities, ensure_ascii=False)
        KNOWN_ANNOTATION_FIXES.append({
            'split': 'train', 'csv_row_number': idx + 2,
            'before': 'شاطئ القويعة في طرابلس',
            'after': corrected_text,
            'action': 'exact surface correction; offsets unchanged',
            'human_review_required': True,
        })
if KNOWN_ANNOTATION_FIXES:
    (OUTPUT / 'known_annotation_fixes.json').write_text(
        json.dumps(KNOWN_ANNOTATION_FIXES, ensure_ascii=False, indent=2),
        encoding='utf-8')
    print('⚠️ Corrected one documented training annotation; see known_annotation_fixes.json')
else:
    print('No documented raw-data correction needed (possibly already using ner_ready dataset).')

REQUIRED={'report','incident_type','priority','location_json','source'}
for s,df in SPLITS.items():
    if not REQUIRED.issubset(df.columns):raise ValueError(f'{s}: missing {REQUIRED-set(df.columns)}')
    if df.empty:raise ValueError(f'{s} is empty')
    print(s,len(df),'real:',sum(df.source.astype(str).str.lower().eq('real')))
print('Data:',DATA_PATH)

⚠️ Corrected one documented training annotation; see known_annotation_fixes.json
train 826 real: 222
val 118 real: 33
test 236 real: 62
Data: /kaggle/input/datasets/ritaj3li3enabdallah/balag/balaagh_dataset


### 5.6 Balaagh QA

Before training, the pipeline checks:
- known label values;
- JSON validity;
- `text[start:end]` equality;
- overlapping entities;
- duplicate/cross-split report leakage.

The report text is not normalized before span validation because changing characters would invalidate the stored location offsets.

In [4]:
def parse_entities(report, payload, split, rownum):
    try: ents=json.loads(payload)
    except Exception as e: raise ValueError(f'{split} row {rownum}: invalid location_json: {e}')
    if not isinstance(ents,list):raise ValueError(f'{split} row {rownum}: entities not list')
    out=[]
    for e in ents:
        st,en,tx,lab=e['start'],e['end'],e['text'],e['label']
        if not (isinstance(st,int) and isinstance(en,int) and 0<=st<en<=len(report) and lab=='LOC'):
            raise ValueError(f'{split} row {rownum}: invalid span or label: {e}')
        if report[st:en]!=tx:raise ValueError(f'{split} row {rownum}: span {e}, actual={report[st:en]!r}')
        out.append({'start':st,'end':en,'text':tx,'label':'LOC'})
    out.sort(key=lambda x:(x['start'],x['end']))
    if any(a['end']>b['start'] for a,b in zip(out,out[1:])):
        raise ValueError(f'{split} row {rownum}: overlapping annotations')
    return out

for split,df in SPLITS.items():
    df['report']=df.report.astype(str)
    df['incident_type']=df.incident_type.astype(str).str.strip()
    df['priority']=df.priority.astype(str).str.strip().str.title()
    df['source']=df.source.astype(str).str.strip().str.lower()
    if (df.report.str.strip()=='').any():raise ValueError(f'{split}: empty report')
    for col,labels in [('incident_type',CATEGORIES),('priority',PRIORITIES)]:
        bad=set(df[col])-set(labels)
        if bad:raise ValueError(f'{split}: unknown {col} labels: {bad}')
    df['entities']=[parse_entities(t,j,split,i+2) for i,(t,j) in enumerate(zip(df.report,df.location_json))]

def norm(t):
    t=re.sub(r'[ًٌٍَُِْـ]','',t).translate(str.maketrans({'أ':'ا','إ':'ا','آ':'ا','ى':'ي'}))
    return re.sub(r'\s+',' ',re.sub(r'[^\w؀-ۿ]+',' ',t)).strip()
keys={s:set(df.report.map(norm)) for s,df in SPLITS.items()}
for a,b in [('train','val'),('train','test'),('val','test')]:
    shared=keys[a]&keys[b]
    print('Exact-normalized cross-split overlap',a,b,len(shared))
    if shared:raise ValueError('Cross-split duplicate text detected; fix before training')
qa={s:{'rows':len(df),'real':int(df.source.eq('real').sum()),'mentions':sum(map(len,df.entities)),
       'without_location':sum(df.entities.map(len)==0)} for s,df in SPLITS.items()}
(OUTPUT/'qa_report.json').write_text(json.dumps(qa,ensure_ascii=False,indent=2),encoding='utf-8')
print('QA passed:',qa)
print('Important: QA checks spans; human annotation review is still needed.')

Exact-normalized cross-split overlap train val 0
Exact-normalized cross-split overlap train test 0
Exact-normalized cross-split overlap val test 0
QA passed: {'train': {'rows': 826, 'real': 222, 'mentions': 746, 'without_location': 107}, 'val': {'rows': 118, 'real': 33, 'mentions': 103, 'without_location': 20}, 'test': {'rows': 236, 'real': 62, 'mentions': 210, 'without_location': 40}}
Important: QA checks spans; human annotation review is still needed.


### 5.7 Select the Stage 1 IDRISI NER checkpoint

In [5]:
def has_weights(folder):
    return folder.is_dir() and (folder/'config.json').exists() and any(
        (folder/x).exists() for x in ('model.safetensors','pytorch_model.bin',
                                      'model.safetensors.index.json','pytorch_model.bin.index.json'))

def is_idrisi_ner(folder):
    if not has_weights(folder):return False
    try:
        cfg=json.loads((folder/'config.json').read_text())
        labels={str(v) for v in cfg.get('id2label',{}).values()}
        return ('B-LOC' in labels and 'I-LOC' in labels and 'O' in labels and
                cfg.get('num_labels',len(labels))==3)
    except Exception:return False

def safe_unzip(src,dest):
    dest.mkdir(parents=True,exist_ok=True)
    with zipfile.ZipFile(src) as z:
        for m in z.infolist():
            target=(dest/m.filename).resolve()
            if not target.is_relative_to(dest.resolve()):raise ValueError('Unsafe ZIP path')
        z.extractall(dest)
    return dest

if MANUAL_IDRISI_MODEL_PATH:
    p=Path(MANUAL_IDRISI_MODEL_PATH)
    if p.suffix=='.zip':
        p=safe_unzip(p,OUTPUT/'unzipped_idrisi')
        models=[q.parent for q in p.rglob('config.json') if is_idrisi_ner(q.parent)]
        if len(models)!=1:raise ValueError(f'Need one IDRISI NER model in ZIP, got: {models}')
        IDRISI_PATH=models[0]
    else:IDRISI_PATH=p
    if not is_idrisi_ner(IDRISI_PATH):raise ValueError('MANUAL_IDRISI_MODEL_PATH is not a 3-label IDRISI NER model')
else:
    # Prefer the named original IDRISI zip; otherwise find a unique LOC-trained final_model.
    archives=all_files('marbert_idrisi_typeless_model.zip')
    if len(archives)>1:raise ValueError(f'Multiple IDRISI archives: {archives}; set MANUAL_IDRISI_MODEL_PATH')
    if archives:
        unpack=safe_unzip(archives[0],OUTPUT/'unzipped_idrisi')
        models=[p.parent for p in unpack.rglob('config.json') if is_idrisi_ner(p.parent)]
        finalists=[p for p in models if p.name=='final_model']
        candidates=finalists or models
    else:
        candidates=[p.parent for root in ROOTS if root.exists() for p in root.rglob('config.json')
                    if is_idrisi_ner(p.parent)]
        finals=[p for p in candidates if p.name=='final_model'
                and 'balag' not in str(p).lower() and 'balaagh' not in str(p).lower()]
        candidates=finals or candidates
    if len(candidates)!=1:
        print('Candidates:',*[str(p) for p in candidates],sep='\n - ')
        raise FileNotFoundError('لازم تحددي MANUAL_IDRISI_MODEL_PATH للنموذج الأصلي الصحيح.')
    IDRISI_PATH=candidates[0]
print('Starting NER model:',IDRISI_PATH)
TOKENIZER=AutoTokenizer.from_pretrained(str(IDRISI_PATH),use_fast=True)
if not TOKENIZER.is_fast:raise RuntimeError('The IDRISI tokenizer must be fast for offset_mapping')
print('Tokenizer:',type(TOKENIZER).__name__)

Starting NER model: /kaggle/working/crisislens_three_tasks/unzipped_idrisi/final_model
Tokenizer: BertTokenizerFast


### 5.8 Build Balaagh NER token labels

In [6]:
ner2id={k:i for i,k in enumerate(NER_LABELS)}

class NERDataset(torch.utils.data.Dataset):
    def __init__(self,frame):
        self.rows=[]
        for row_i,r in frame.reset_index(drop=True).iterrows():
            text=r['report']; ents=r['entities']
            enc=TOKENIZER(text,truncation=True,max_length=MAX_LENGTH,return_offsets_mapping=True)
            offsets=enc.pop('offset_mapping')
            labels=[]; covered=[False]*len(ents); previous_ent=-1
            for (st,en) in offsets:
                if st==en:
                    labels.append(-100);continue
                matches=[i for i,e in enumerate(ents) if en>e['start'] and st<e['end']]
                if len(matches)>1:raise ValueError(f'Overlapping token matches in NER at row {row_i}')
                if not matches:
                    labels.append(ner2id['O']);previous_ent=-1
                else:
                    idx=matches[0];covered[idx]=True
                    labels.append(ner2id['I-LOC'] if previous_ent==idx else ner2id['B-LOC'])
                    previous_ent=idx
            if ents and not all(covered):raise ValueError(f'MAX_LENGTH truncates location at row {row_i}; increase MAX_LENGTH')
            # Additional guard: span must be fully inside last retained real token.
            end_of_tokens=max([en for st,en in offsets if en>st],default=0)
            if any(e['end']>end_of_tokens for e in ents):
                raise ValueError(f'MAX_LENGTH cuts LOC at row {row_i}')
            enc['labels']=labels
            self.rows.append(enc)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):return self.rows[i]

NER_DATA={s:NERDataset(df) for s,df in SPLITS.items()}
NER_REAL_TEST=NERDataset(SPLITS['test'][SPLITS['test'].source.eq('real')])
print('NER tokenized:',{k:len(v) for k,v in NER_DATA.items()})

NER tokenized: {'train': 826, 'val': 118, 'test': 236}


In [7]:
def bio_spans(tags):
    result=[];start=None
    for i,tag in enumerate(list(tags)+['O']):
        if tag=='B-LOC':
            if start is not None:result.append((start,i))
            start=i
        elif tag=='I-LOC':
            if start is None:start=i
        else:
            if start is not None:result.append((start,i))
            start=None
    return result

def ner_score(predictions,labels):
    predicted=np.asarray(predictions)
    if predicted.ndim==3:predicted=np.argmax(predicted,axis=-1)
    tp=fp=fn=tot=correct=0
    for p_row,g_row in zip(predicted,labels):
        pp=[NER_LABELS[int(p)] for p,g in zip(p_row,g_row) if g!=-100]
        gg=[NER_LABELS[int(g)] for g in g_row if g!=-100]
        p,g=set(bio_spans(pp)),set(bio_spans(gg))
        tp+=len(p&g);fp+=len(p-g);fn+=len(g-p)
        correct+=sum(x==y for x,y in zip(pp,gg));tot+=len(gg)
    prec=tp/(tp+fp) if tp+fp else 0.0
    rec=tp/(tp+fn) if tp+fn else 0.0
    return {'precision':prec,'recall':rec,'f1':2*prec*rec/(prec+rec) if prec+rec else 0.,
            'token_accuracy':correct/tot if tot else 0.,'tp':tp,'fp':fp,'fn':fn}

def ner_metrics(pred):
    m=ner_score(pred.predictions,pred.label_ids)
    return {k:v for k,v in m.items() if k in ('precision','recall','f1','token_accuracy')}

print('Metric ready. F1 = exact matching token-level entity spans, not char-level F1.')

Metric ready. F1 = exact matching token-level entity spans, not char-level F1.


### 5.9 Continue Location NER training on Balaagh

In [8]:
NER_DIR=OUTPUT/'location_ner';NER_FINAL=NER_DIR/'final_model'

if RUN_TRAINING:
    ner_model=AutoModelForTokenClassification.from_pretrained(str(IDRISI_PATH))
    args=TrainingArguments(
        output_dir=str(NER_DIR/'checkpoints'),learning_rate=NER_LR,
        per_device_train_batch_size=TRAIN_BATCH,per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,num_train_epochs=NER_EPOCHS,
        weight_decay=0.01,warmup_ratio=0.1,eval_strategy='epoch',save_strategy='epoch',
        load_best_model_at_end=True,metric_for_best_model='f1',greater_is_better=True,
        fp16=torch.cuda.is_available(),save_total_limit=2,logging_steps=25,
        report_to='none',seed=SEED)
    trainer=Trainer(model=ner_model,args=args,train_dataset=NER_DATA['train'],
        eval_dataset=NER_DATA['val'],processing_class=TOKENIZER,
        data_collator=DataCollatorForTokenClassification(TOKENIZER),compute_metrics=ner_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    NER_FINAL.mkdir(parents=True,exist_ok=True)
    trainer.save_model(str(NER_FINAL));TOKENIZER.save_pretrained(str(NER_FINAL))
    print('Best IDRISI→Balaagh NER checkpoint:',trainer.state.best_model_checkpoint)
    print('Validation:',trainer.evaluate())
    del trainer,ner_model
    gc.collect();torch.cuda.empty_cache()
print('Saved NER:',NER_FINAL)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Token Accuracy
1,0.201900,0.106126,0.710526,0.786408,0.746544,0.960784
2,0.083900,0.103097,0.803738,0.834951,0.819048,0.965359
3,0.054100,0.104663,0.838095,0.854369,0.846154,0.967974
4,0.048900,0.106027,0.822430,0.854369,0.838095,0.968627


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Best IDRISI→Balaagh NER checkpoint: /kaggle/working/crisislens_three_tasks/location_ner/checkpoints/checkpoint-78


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Validation: {'eval_loss': 0.10466321557760239, 'eval_precision': 0.8380952380952381, 'eval_recall': 0.8543689320388349, 'eval_f1': 0.8461538461538461, 'eval_token_accuracy': 0.9679738562091503, 'eval_runtime': 0.4151, 'eval_samples_per_second': 284.295, 'eval_steps_per_second': 9.637, 'epoch': 4.0}
Saved NER: /kaggle/working/crisislens_three_tasks/location_ner/final_model


### 5.10 Prepare category and priority sequence-classification datasets

In [ ]:
class ClassificationDataset(torch.utils.data.Dataset):
    def __init__(self,frame,target,names):
        self.labels=[names.index(x) for x in frame[target].tolist()]
        self.enc=TOKENIZER(frame.report.tolist(),truncation=True,max_length=MAX_LENGTH)
    def __len__(self):return len(self.labels)
    def __getitem__(self,i):
        return {**{k:v[i] for k,v in self.enc.items()},'labels':self.labels[i]}

def classification_metrics(pred):
    guess=np.argmax(pred.predictions,axis=1);gold=pred.label_ids
    p,r,f1,_=precision_recall_fscore_support(gold,guess,average='macro',zero_division=0)
    weighted=precision_recall_fscore_support(gold,guess,average='weighted',zero_division=0)[2]
    return {'macro_f1':float(f1),'weighted_f1':float(weighted),
            'precision':float(p),'recall':float(r),'accuracy':float(accuracy_score(gold,guess))}

CLASS_TASKS=[('incident_category','incident_type',CATEGORIES),('priority','priority',PRIORITIES)]
CLASS_FINAL={name:OUTPUT/name/'final_model' for name,_,_ in CLASS_TASKS}
COLLATOR=DataCollatorWithPadding(TOKENIZER)

### 5.11 Train Incident Category and Priority models

In [10]:
def fit_classification(task,col,names):
    task_dir=OUTPUT/task;final=task_dir/'final_model'
    data={s:ClassificationDataset(df,col,names) for s,df in SPLITS.items()}
    model=AutoModelForSequenceClassification.from_pretrained(
        str(IDRISI_PATH),num_labels=len(names),
        id2label={i:x for i,x in enumerate(names)},label2id={x:i for i,x in enumerate(names)},
        ignore_mismatched_sizes=True)
    # Expected: classification head initializes from scratch; encoder is reused.
    args=TrainingArguments(
        output_dir=str(task_dir/'checkpoints'),learning_rate=CLASS_LR,
        per_device_train_batch_size=TRAIN_BATCH,per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,num_train_epochs=CLASS_EPOCHS,
        weight_decay=0.01,warmup_ratio=0.1,eval_strategy='epoch',save_strategy='epoch',
        load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,
        fp16=torch.cuda.is_available(),save_total_limit=2,logging_steps=25,
        report_to='none',seed=SEED)
    trainer=Trainer(model=model,args=args,train_dataset=data['train'],eval_dataset=data['val'],
                    processing_class=TOKENIZER,data_collator=COLLATOR,
                    compute_metrics=classification_metrics,
                    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    final.mkdir(parents=True,exist_ok=True)
    trainer.save_model(str(final));TOKENIZER.save_pretrained(str(final))
    print(task,'best checkpoint:',trainer.state.best_model_checkpoint)
    print('validation:',trainer.evaluate())
    del trainer,model,data
    gc.collect();torch.cuda.empty_cache()

if RUN_TRAINING:
    for task,col,names in CLASS_TASKS:fit_classification(task,col,names)
print('Finished three tasks:',NER_FINAL,*CLASS_FINAL.values(),sep='\n - ')

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /kaggle/working/crisislens_three_tasks/unzipped_idrisi/final_model and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /kaggle/working/crisislens_three_tasks/unzipped_idrisi/final_model and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([3]) in the checkpoint and torch.Size([6]) in the model instantiated
- classifier.weight: found shape torch.Size([3, 768]) in the checkpoint and torch.Size([6, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: User

Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1,Precision,Recall,Accuracy
1,1.700100,1.317935,0.565221,0.598984,0.567859,0.587938,0.627119
2,1.001500,0.716457,0.761192,0.775938,0.801656,0.754308,0.779661
3,0.538500,0.536991,0.831926,0.839201,0.838000,0.828750,0.838983
4,0.335900,0.490467,0.832933,0.838628,0.841434,0.828389,0.838983
5,0.244000,0.468931,0.843457,0.848641,0.847680,0.844623,0.847458


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

incident_category best checkpoint: /kaggle/working/crisislens_three_tasks/incident_category/checkpoints/checkpoint-130


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1,Precision,Recall,Accuracy
1,1.341400,1.249786,0.373038,0.409401,0.346951,0.414749,0.457627
2,1.109600,1.010747,0.505056,0.516879,0.555166,0.492728,0.525424
3,0.846200,0.901137,0.545303,0.550023,0.589599,0.531981,0.550847
4,0.650700,0.859599,0.550270,0.565361,0.567135,0.552304,0.576271
5,0.540400,0.864683,0.550245,0.567098,0.622010,0.549301,0.584746


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector

priority best checkpoint: /kaggle/working/crisislens_three_tasks/priority/checkpoints/checkpoint-104


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


validation: {'eval_loss': 0.8595993518829346, 'eval_macro_f1': 0.5502696015601831, 'eval_weighted_f1': 0.565361230671823, 'eval_precision': 0.5671351674641149, 'eval_recall': 0.5523037278152367, 'eval_accuracy': 0.576271186440678, 'eval_runtime': 0.4028, 'eval_samples_per_second': 292.94, 'eval_steps_per_second': 9.93, 'epoch': 5.0}
Finished three tasks:
 - /kaggle/working/crisislens_three_tasks/location_ner/final_model
 - /kaggle/working/crisislens_three_tasks/incident_category/final_model
 - /kaggle/working/crisislens_three_tasks/priority/final_model


### Training / Processing Observations

Recorded best validation results from the Balaagh stage:

| Task | Best Epoch / Checkpoint | Best Validation Metric |
|---|---:|---:|
| Location NER | Epoch 3 | F1 = **0.8462** |
| Incident Category | Epoch 5 | Macro F1 = **0.8435** |
| Priority | Epoch 4 | Macro F1 = **0.5503** |

The category task learned strongly, while priority was noticeably harder. This matches the later test behavior and indicates that priority labels and boundary cases require additional human review.

## 6. Evaluation & Results

### 6.1 Stage 1 validation

The IDRISI model is evaluated on its Dev split using **exact entity-level** Precision, Recall and F1.

### 6.2 Stage 2 test

The Balaagh test set is held out from training.

Two evaluation subsets are reported:

- `test_all`: all 236 held-out Balaagh reports.
- `test_real`: the 62 test reports marked as real.

### 6.3 Location before/after comparison

The comparison evaluates:
1. **IDRISI_only**
2. **IDRISI_plus_Balaagh**

Both are evaluated on the same Balaagh test examples.

### 6.4 Stage 1 Dev evaluation

In [15]:
dev_metrics = trainer.evaluate(dev_ds)

print("=== DEV METRICS ===")
for k, v in dev_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(k, ":", v)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


=== DEV METRICS ===
eval_loss: 0.0287
eval_precision: 0.8300
eval_recall: 0.8489
eval_f1: 0.8393
eval_token_accuracy: 0.9909
eval_runtime: 1.8620
eval_samples_per_second: 248.1240
eval_steps_per_second: 4.2970
epoch: 5.0000


In [16]:

pred_output = trainer.predict(dev_ds)
pred_ids = np.argmax(pred_output.predictions, axis=-1)
gold_ids = pred_output.label_ids

pred_tag_sequences = []
gold_tag_sequences = []

for pred_row, gold_row in zip(pred_ids, gold_ids):
    p_seq = []
    g_seq = []

    for p, g in zip(pred_row, gold_row):
        if g == -100:
            continue
        p_seq.append(id2label[int(p)])
        g_seq.append(id2label[int(g)])

    pred_tag_sequences.append(p_seq)
    gold_tag_sequences.append(g_seq)

detailed = entity_metrics(gold_tag_sequences, pred_tag_sequences)

print("=== EXACT ENTITY-LEVEL REPORT ===")
print(f"True Positives : {detailed['tp']}")
print(f"False Positives: {detailed['fp']}")
print(f"False Negatives: {detailed['fn']}")
print(f"Precision      : {detailed['precision']:.4f}")
print(f"Recall         : {detailed['recall']:.4f}")
print(f"F1             : {detailed['f1']:.4f}")
print(f"Token Accuracy : {detailed['token_accuracy']:.4f}")


=== EXACT ENTITY-LEVEL REPORT ===
True Positives : 410
False Positives: 84
False Negatives: 73
Precision      : 0.8300
Recall         : 0.8489
F1             : 0.8393
Token Accuracy : 0.9909


### 6.5 Compare location extraction before and after Balaagh fine-tuning

In [11]:
def score_ner_checkpoint(path,dataset):
    model=AutoModelForTokenClassification.from_pretrained(str(path))
    args=TrainingArguments(output_dir=str(OUTPUT/'tmp_ner_eval'),per_device_eval_batch_size=EVAL_BATCH,
        fp16=torch.cuda.is_available(),report_to='none')
    evaluator=Trainer(model=model,args=args,data_collator=DataCollatorForTokenClassification(TOKENIZER),
                      processing_class=TOKENIZER)
    pred=evaluator.predict(dataset)
    scores=ner_score(pred.predictions,pred.label_ids)
    del evaluator,model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return scores

ner_comparison=[]
for model_label,path in [('IDRISI_only',IDRISI_PATH),('IDRISI_plus_Balaagh',NER_FINAL)]:
    for subset,ds in [('test_all',NER_DATA['test']),('test_real',NER_REAL_TEST)]:
        if not len(ds):continue
        m=score_ner_checkpoint(path,ds)
        ner_comparison.append({'model':model_label,'subset':subset,**m})
comparison_df=pd.DataFrame(ner_comparison)
display(comparison_df)
comparison_df.to_csv(OUTPUT/'ner_before_after.csv',index=False)
print('NER scores saved')

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


,model,subset,precision,recall,f1,token_accuracy,tp,fp,fn
0,IDRISI_only,test_all,0.481061,0.604762,0.535865,0.910078,127,137,83
1,IDRISI_only,test_real,0.333333,0.571429,0.421053,0.832455,36,72,27
2,IDRISI_plus_Balaagh,test_all,0.788546,0.852381,0.819222,0.969302,179,48,31
3,IDRISI_plus_Balaagh,test_real,0.631579,0.761905,0.690647,0.934668,48,28,15


NER scores saved


### 6.6 Evaluate Category and Priority on the held-out test set

In [ ]:
# V4: evaluation of both sequence-classification models on the same test set.
# All indentation here uses four spaces, never tabs.
CLASS_RESULTS = {}

for task, col, names in CLASS_TASKS:
    model = AutoModelForSequenceClassification.from_pretrained(str(CLASS_FINAL[task]))
    args = TrainingArguments(
        output_dir=str(OUTPUT / f"tmp_{task}"),
        per_device_eval_batch_size=EVAL_BATCH,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    evaluator = Trainer(
        model=model,
        args=args,
        data_collator=COLLATOR,
        processing_class=TOKENIZER,
    )

    data = SPLITS["test"]
    test_ds = ClassificationDataset(data, col, names)
    pred = evaluator.predict(test_ds)
    gold = np.asarray(pred.label_ids)
    guess = np.argmax(pred.predictions, axis=-1)
    probs = torch.softmax(torch.tensor(pred.predictions), dim=-1).numpy()
    scores = {}

    # No hanging/continuation indentation in the subset loop.
    subsets = {
        "test_all": np.ones(len(data), dtype=bool),
        "test_real": data["source"].eq("real").to_numpy(),
    }
    for subset, mask in subsets.items():
        if not mask.any():
            continue

        report = classification_report(
            gold[mask],
            guess[mask],
            labels=list(range(len(names))),
            target_names=names,
            zero_division=0,
            output_dict=True,
        )
        cm = confusion_matrix(
            gold[mask], guess[mask], labels=list(range(len(names)))
        )
        scores[subset] = {
            "classification_report": report,
            "confusion_matrix": cm.tolist(),
        }
        print(task, subset, "macro-F1:", round(report["macro avg"]["f1-score"], 4), "n:", int(mask.sum()))

    failures = pd.DataFrame({
        "report": data["report"].to_numpy(),
        "source": data["source"].to_numpy(),
        "actual": [names[int(x)] for x in gold],
        "predicted": [names[int(x)] for x in guess],
        "confidence": probs.max(axis=1).round(4),
    })
    if task == "priority":
        failures["distance"] = (guess - gold).astype(int)
        failures["error_type"] = failures["distance"].map(
            lambda d: "Correct" if d == 0 else
            "Adjacent (review required)" if abs(d) == 1 else "Major error"
        )
        failures["underestimated"] = failures["distance"] < 0
    else:
        failures["error_type"] = np.where(gold == guess, "Correct", "Wrong class")

    failures.to_csv(
        OUTPUT / f"{task}_test_predictions.csv",
        index=False,
        encoding="utf-8-sig",
    )
    CLASS_RESULTS[task] = scores
    if task == "priority":
        print("Priority errors (first 25):")
        display(
            failures.loc[failures["actual"] != failures["predicted"],
                         ["report", "actual", "predicted", "error_type", "underestimated"]].head(25)
        )

    del evaluator, model, test_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

(OUTPUT / "classification_test_metrics.json").write_text(
    json.dumps(CLASS_RESULTS, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Classification evaluation completed; saved CSV and JSON results.")


### Result Interpretation

#### Location NER — same Balaagh test set

| Model | Subset | Precision | Recall | F1 |
|---|---|---:|---:|---:|
| IDRISI only | Test all | 0.4811 | 0.6048 | **0.5359** |
| IDRISI + Balaagh | Test all | 0.7885 | 0.8524 | **0.8192** |
| IDRISI only | Real only | 0.3333 | 0.5714 | **0.4211** |
| IDRISI + Balaagh | Real only | 0.6316 | 0.7619 | **0.6906** |

Libya-specific fine-tuning improved entity F1 by:
- **+28.34 percentage points** on the full Balaagh test set.
- **+26.96 percentage points** on the real-only subset.

#### Incident Category

- Test-all Accuracy: **0.9025**
- Test-all Macro F1: **0.8986**
- Real-only Accuracy: **0.9516**
- Real-only Macro F1: **0.9417**

#### Priority

- Test-all Accuracy: **0.6017**
- Test-all Macro F1: **0.5990**
- Real-only Accuracy: **0.5968**
- Real-only Macro F1: **0.5391**

The largest remaining limitation is Priority classification.

## 7. Error Analysis

### Location extraction
After Balaagh fine-tuning:
- Test-all TP increased from **127 → 179**.
- FP decreased from **137 → 48**.
- FN decreased from **83 → 31**.

On real reports:
- TP increased from **36 → 48**.
- FP decreased from **72 → 28**.
- FN decreased from **27 → 15**.

### Priority
Priority labels are ordinal:

`Low → Medium → High → Critical`

A one-level difference is recorded as an adjacent error for analysis, but it is not automatically safe or acceptable. Underestimating a Critical case is particularly important.

### Error-analysis outputs

The Stage 2 evaluation exports:
- `priority_test_predictions.csv`
- `incident_category_test_predictions.csv`
- confusion matrices inside `classification_test_metrics.json`

## 8. Sample Predictions / Outputs

The following function loads the three final Stage 2 models and accepts one report as input.

It returns:
- explicit location mentions;
- incident category;
- priority.

The three outputs come from three task-specific final models that share the same IDRISI-trained MARBERT lineage.

In [15]:
import torch.nn.functional as F
LOADED={}

def load_for_demo():
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if LOADED:return
    LOADED['device']=device
    LOADED['ner']=AutoModelForTokenClassification.from_pretrained(str(NER_FINAL)).eval().to(device)
    for task,_,_ in CLASS_TASKS:
        LOADED[task]=AutoModelForSequenceClassification.from_pretrained(str(CLASS_FINAL[task])).eval().to(device)

def predict_report(text):
    if not isinstance(text,str) or not text.strip():raise ValueError('أدخلي نص البلاغ')
    load_for_demo()
    device=LOADED['device'];result={'report':text}
    for task,_,names in CLASS_TASKS:
        tokenized=TOKENIZER(text,return_tensors='pt',truncation=True,max_length=MAX_LENGTH)
        tokenized={k:v.to(device) for k,v in tokenized.items()}
        with torch.inference_mode():
            probs=F.softmax(LOADED[task](**tokenized).logits,dim=-1)[0].cpu().numpy()
        idx=int(np.argmax(probs))
        result[task]={'label':names[idx],'confidence':round(float(probs[idx]),4)}
    enc=TOKENIZER(text,return_tensors='pt',return_offsets_mapping=True,truncation=True,
                  max_length=MAX_LENGTH)
    offsets=enc.pop('offset_mapping')[0].tolist()
    inp={k:v.to(device) for k,v in enc.items()}
    with torch.inference_mode():
        probs=F.softmax(LOADED['ner'](**inp).logits,dim=-1)[0].cpu().numpy()
    ids=np.argmax(probs,axis=-1)
    mentions=[];current=None
    # Map predicted BIO spans back into character positions.
    for (st,en),label_id,token_probs in zip(offsets,ids,probs):
        if st==en:continue
        tag=LOADED['ner'].config.id2label[int(label_id)]
        if tag=='B-LOC':
            if current:mentions.append(current)
            current={'start':st,'end':en,'confidence_values':[float(token_probs[label_id])]}
        elif tag=='I-LOC':
            if current is None:current={'start':st,'end':en,'confidence_values':[]}
            current['end']=en;current['confidence_values'].append(float(token_probs[label_id]))
        else:
            if current:mentions.append(current);current=None
    if current:mentions.append(current)
    result['locations']=[{'text':text[e['start']:e['end']], 'start':e['start'], 'end':e['end'],
                          'confidence':round(float(np.mean(e['confidence_values'])),4)}
                         for e in mentions]
    return result

example='حادث على طريق المطار والسائق محاصر داخل السيارة، نبو إسعاف بسرعة.'
print(json.dumps(predict_report(example),ensure_ascii=False,indent=2))

# في Kaggle تقدرِي تستخدمي: predict_report('بلاغك هنا')

{
  "report": "حادث على طريق المطار والسائق محاصر داخل السيارة، نبو إسعاف بسرعة.",
  "incident_category": {
    "label": "Road/Transportation",
    "confidence": 0.9251
  },
  "priority": {
    "label": "High",
    "confidence": 0.559
  },
  "locations": [
    {
      "text": "طريق المطار",
      "start": 9,
      "end": 20,
      "confidence": 0.9933
    }
  ]
}


### Optional interactive Kaggle input

In [16]:
try:
    import ipywidgets as widgets
    from IPython.display import display,clear_output
    box=widgets.Textarea(value='انقطاع كهرباء في تاجوراء من ساعات',description='بلاغ:',
                         layout=widgets.Layout(width='95%',height='100px'))
    run=widgets.Button(description='تحليل البلاغ',button_style='success')
    output=widgets.Output()
    def on_click(_):
        with output:
            clear_output()
            try:print(json.dumps(predict_report(box.value),ensure_ascii=False,indent=2))
            except Exception as exc:print(type(exc).__name__,exc)
    run.on_click(on_click)
    display(box,run,output)
except ImportError:
    print('ipywidgets unavailable; use predict_report(text) above.')

Textarea(value='انقطاع كهرباء في تاجوراء من ساعات', description='بلاغ:', layout=Layout(height='100px', width='…

Button(button_style='success', description='تحليل البلاغ', style=ButtonStyle())

Output()

### Save final task models and evaluation artifacts

In [17]:
# Unload inference models before packaging
for k,v in list(LOADED.items()):
    if k!='device':v.to('cpu')
LOADED.clear();gc.collect()
if torch.cuda.is_available():torch.cuda.empty_cache()

README={
  'project':'CrisisLens Libya','base_model':str(IDRISI_PATH),
  'tasks':{'location_ner':'O / B-LOC / I-LOC','incident_category':CATEGORIES,'priority':PRIORITIES},
  'source_data':str(DATA_PATH),'caveat':'Provisional metrics; synthetic templates and human label review limits.'
}
(OUTPUT/'README.json').write_text(json.dumps(README,ensure_ascii=False,indent=2),encoding='utf-8')
archive=Path('/kaggle/working/crisislens_marbert_three_tasks.zip')
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
    for task,folder in [('location_ner',NER_FINAL),*[(x,CLASS_FINAL[x]) for x in CLASS_FINAL]]:
        if not (folder/'config.json').exists():raise FileNotFoundError(folder)
        for p in folder.rglob('*'):
            if p.is_file():z.write(p,arcname=str(Path(task)/'final_model'/p.relative_to(folder)))
    for filename in ['qa_report.json','ner_before_after.csv','classification_test_metrics.json',
                     'incident_category_test_predictions.csv','priority_test_predictions.csv','README.json']:
        p=OUTPUT/filename
        if p.exists():z.write(p,arcname=filename)
print('Created:',archive,'size MB:',round(archive.stat().st_size/(1024**2),1))
from IPython.display import FileLink,display
display(FileLink(str(archive)))

Created: /kaggle/working/crisislens_marbert_three_tasks.zip size MB: 1736.8


/kaggle/working/crisislens_marbert_three_tasks.zip

## 9. Conclusion

The CrisisLens Libya MARBERT pipeline was built through **two sequential training stages**, preceded by IDRISI cleaning.

### Stage 0 — IDRISI preparation
The public IDRISI Gold data was converted into a clean type-less Location NER dataset with **3,696 tweets and 3,932 valid location mentions**.

### Stage 1 — IDRISI Location NER
MARBERT was fine-tuned for Arabic Location extraction and achieved **0.8393 Dev entity F1**.

### Stage 2 — Balaagh adaptation
The IDRISI-trained checkpoint was reused:
- the NER model was further fine-tuned on Libyan reports;
- a new Incident Category classifier was trained from the IDRISI-trained encoder;
- a new Priority classifier was trained from the same encoder.

Held-out results:
- **Location NER F1: 0.8192**
- **Incident Category Macro F1: 0.8986**
- **Priority Macro F1: 0.5990**

Location F1 increased from **0.5359 to 0.8192** on the same Balaagh test set after Libya-specific adaptation.

### Final lineage

```text
Raw IDRISI Gold
      ↓
IDRISI cleaning / schema correction
      ↓
MARBERT + IDRISI Location NER
      ↓
IDRISI-trained MARBERT checkpoint
      ├── Balaagh Location NER
      ├── Balaagh Incident Category
      └── Balaagh Priority
```

This notebook documents the separate three-task experiment. The later unified multi-task model is a distinct experiment.